# 07 · JSBSim through Python, and where Project Chrono would fit

JSBSim is an optional second physics backend: an independent C++ engine that integrates the same aircraft, used to cross-check the Python integrator, frames and ground contact. Project Chrono is not used in this repository; the second half of this notebook explains where it would fit.

In [1]:
# Setup: make the repository importable and load the V3 airframe used throughout these notebooks
import sys, math, pathlib
ROOT = pathlib.Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
from airframe_designer.geometry.airframe import Airframe

np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({"figure.figsize": (7, 3.6), "axes.grid": True, "grid.alpha": 0.3})

AF_PATH = ROOT / "airframes" / "atlas_v3_v34_foils_50_65_50_tuned.json"
af = Airframe.load(AF_PATH)
af.resolve_mass()
G = 9.80665
print(f"{af.name}: {af.mass.mass:.2f} kg, {len(af.active_rotors())} fans, hover pitch {af.hover_pitch_deg} deg, parked {af.landed_pitch_deg} deg")

ATLAS_V3_V34_FOILS_50_65_50_TUNED: 11.83 kg, 9 fans, hover pitch 23.85 deg, parked 23.85 deg


## 1. What JSBSim does and what Python still does

| Task | Done by |
|---|---|
| 6-DOF integration, gravity, Earth model (rotating WGS84) | JSBSim |
| Atmosphere | JSBSim (the Python backend uses constant density) |
| Leg contact (feet as `STRUCTURE` contacts) | JSBSim |
| Moment arms of the fan forces | JSBSim, from each force's location |
| Fan spool, thrust, reaction torque, ram drag, body drag | Python, injected every step |
| Wings, if any | JSBSim, from tables Python generates |

> **Note: why a second engine is useful**
>
> If two independently written integrators fly the same aircraft with the same forces and agree, the frames, signs, contact model and integration are probably right. JSBSim does not check the aerodynamics, because both backends use the same fan and drag code.

## 2. Generating the JSBSim aircraft file

**Theory.** `generate_model` writes an aircraft XML: mass and inertia (slugs, slug·ft²), each foot as a contact, one `external_reactions` force per fan in the BODY frame at the fan's position, and aerodynamic tables. JSBSim's structural frame is x aft, y right, z up, in inches, so every point maps as $\mathbf r_{\text{JSB}} = (-x,\ y,\ -z)\times 39.37$.

In [2]:
from airframe_designer.dynamics.jsbsim_backend import generate_model, JSB_ROOT
mdir = generate_model(af, "notebook_demo")
xml = (mdir / "notebook_demo.xml").read_text()
print("written to", mdir)
print(xml[:1800])

written to /Users/stefanosfragkoulis/.airframe_designer/jsbsim/aircraft/notebook_demo
<?xml version="1.0"?>
<fdm_config name="notebook_demo" version="2.0" release="ALPHA">
<fileheader><author>airframe_designer</author><description>generated from ATLAS_V3_V34_FOILS_50_65_50_TUNED</description></fileheader>
<metrics><wingarea unit="M2">1e-06</wingarea><wingspan unit="M">1</wingspan><chord unit="M">1e-06</chord><htailarea unit="M2">0</htailarea><htailarm unit="M">0</htailarm><vtailarea unit="M2">0</vtailarea><vtailarm unit="M">0</vtailarm><location name="AERORP" unit="IN"><x>3.72992</x><y>-0.157087</y><z>-3.10433</z></location><location name="EYEPOINT" unit="IN"><x>3.72992</x><y>-0.157087</y><z>-3.10433</z></location><location name="VRP" unit="IN"><x>0</x><y>0</y><z>0</z></location></metrics>
<mass_balance><ixx unit="SLUG*FT2">0.420034</ixx><iyy unit="SLUG*FT2">0.682241</iyy><izz unit="SLUG*FT2">0.96474</izz><ixy unit="SLUG*FT2">0.00452789</ixy><ixz unit="SLUG*FT2">-0.0620194</ixz><iyz un

In [3]:
# The unit and axis conversion, by hand, for the first fan
r0 = af.active_rotors()[0]
p = np.asarray(r0.pos, float)
print("fan position, ours (FRD, m):", p, " -> JSBSim (aft, right, up, in):", np.round(np.array([-p[0], p[1], -p[2]]) * 39.37, 3))
print("mass", af.mass.mass, "kg =", round(af.mass.mass / 14.5939, 4), "slug;  Ixx", af.mass.inertia[0], "kg m^2 =",
      round(af.mass.inertia[0] / 1.35582, 4), "slug ft^2")

fan position, ours (FRD, m): [-0.5401 -0.413   0.3719]  -> JSBSim (aft, right, up, in): [ 21.264 -16.26  -14.642]
mass 11.83 kg = 0.8106 slug;  Ixx 0.569489 kg m^2 = 0.42 slug ft^2


## 3. FGFDMExec: driving JSBSim from Python

**Theory.** `FGFDMExec` ("Flight Dynamics Model Executive") is JSBSim's main object. One instance is one complete simulated aircraft. It reads the XML, owns all the sub-models (mass, propulsion, aerodynamics, ground reactions, external forces, atmosphere, Earth model, integrator), runs them in order every step, and keeps a **property tree** of named values such as `velocities/p-rad_sec`, whose suffix gives the unit.

> **Note: the four calls the backend makes**
>
> 1. `fdm = jsbsim.FGFDMExec(root)` creates the executive.
> 2. `fdm.load_model(name)` and `fdm.set_dt(dt)` load the aircraft and set the step.
> 3. `fdm["ic/theta-deg"] = pitch` (and latitude, height, heading), then `fdm.run_ic()` applies the initial conditions.
> 4. Every step: write each fan's thrust into `external_reactions/rotor<i>/magnitude`, call `fdm.run()`, read back properties such as `velocities/v-north-fps`.
>
> The cell runs this only if the `jsbsim` package is installed; on this Mac it is not, so it prints how to install it.

In [4]:
try:
    import jsbsim
except ImportError:
    jsbsim = None
    print("jsbsim is not installed in this environment. Install it with:")
    print("    uv pip install --python .venv/bin/python jsbsim")
if jsbsim is not None:
    fdm = jsbsim.FGFDMExec(str(JSB_ROOT))
    fdm.set_debug_level(0)
    assert fdm.load_model("notebook_demo")
    fdm.set_dt(1e-3)
    fdm["ic/h-agl-ft"] = 0.5; fdm["ic/theta-deg"] = af.landed_pitch_deg
    fdm.run_ic()
    for _ in range(2000):
        fdm.run()
    print("after 2 s at rest: theta", round(fdm["attitude/theta-deg"], 2), "deg, h-agl", round(fdm["position/h-agl-ft"] * 0.3048, 3), "m")
    # the same through the simulator's backend class
    from airframe_designer.dynamics.jsbsim_backend import JSBSimBody
    jb = JSBSimBody(af); jb.set_motor_commands(np.zeros(len(af.active_rotors())))
    for _ in range(2000):
        jb.step(1e-3)
    print("JSBSimBody pitch", round(math.degrees(jb.euler[1]), 2), "deg")

jsbsim is not installed in this environment. Install it with:
    uv pip install --python .venv/bin/python jsbsim


## 4. Project Chrono: what it is and where it would fit

**Theory.** Project Chrono is an open-source multibody dynamics library (C++, Python bindings called PyChrono). It is built around many connected bodies and their contacts:

- rigid bodies with joints (revolute, prismatic, springs and dampers), solved as a constrained system;
- contact as penalty forces with Hertz or Hunt-Crossley laws (SMC), $f_n = k\delta^{n} + c\,\delta^{n}\dot\delta$, or as hard complementarity constraints (NSC);
- flexible bodies through finite elements (beams, shells);
- add-ons for vehicles, granular terrain and fluid-structure interaction.

It has no aircraft aerodynamics: the fan and drag forces would still come from this project's Python code, as with JSBSim.

| Area | Today | With Chrono |
|---|---|---|
| Ground contact | point feet, linear spring-damper, regularised friction | real foot shapes, Hunt-Crossley contact, stick-slip friction |
| Nose-lift rotation | pivots on penalty springs at the rear feet | a pivot held as a constraint, no spring bounce |
| Legs | rigid lines, re-solved per parked angle | articulated or compliant legs with their own mass |
| Structure | analytic vibration modes feeding the IMU only | flexible booms and mounts whose modes feed back into the flight |

> **Note: is it worth adding**
>
> Only if the ground phases (park, fan-driven rotation, touchdown, nose-lowering) turn out to be the limiting factor. In the air Chrono adds nothing over the current rigid-body integrator. A practical route mirrors the JSBSim bridge: one Chrono body with the vehicle's mass and inertia, foot contact shapes, the Python `RotorSet` applying forces through Chrono's force accumulators each step, and `DoStepDynamics` at 1 ms; then fly the same scenarios on both and compare the ground phases. The sketch below shows the shape of that code; it only runs if `pychrono` is installed.

In [5]:
try:
    import pychrono as chrono
except ImportError:
    chrono = None
    print("pychrono is not installed (it is distributed through conda: conda install -c projectchrono pychrono).")
if chrono is not None:
    sys_ = chrono.ChSystemSMC()
    sys_.SetGravitationalAcceleration(chrono.ChVector3d(0, 0, -G))      # Chrono is usually z-up
    body = chrono.ChBody()
    body.SetMass(af.mass.mass)
    body.SetInertiaXX(chrono.ChVector3d(*af.mass.inertia))
    sys_.Add(body)
    print("Chrono body created; next steps: foot contact shapes, force accumulators fed by RotorSet, DoStepDynamics(1e-3)")

pychrono is not installed (it is distributed through conda: conda install -c projectchrono pychrono).
